# Reproducing Pagan et al. (2025) from dandiset 001550

**Paper.** Pagan M, Tang VD, Aoi MC, Pillow JW, Mante V, Sussillo D, Brody CD (2025). Individual
variability of neural computations underlying flexible decisions. *Nature* 639, 421-429.
[doi:10.1038/s41586-024-08433-6](https://doi.org/10.1038/s41586-024-08433-6). This attempt used
the open access full text (PubMed Central PMC11903320, CC BY), with the Methods and the Extended
Data legends.

**Dandiset.** [DANDI:001550](https://dandiarchive.org/dandiset/001550/0.260916.0811), version
0.260916.0811. The paper cites no DANDI version: its data statement points to a GitHub repository,
and the dandiset was made later from the same behavioural files. This is the only published
version, so it is pinned; the draft has the same 16,115 assets and size.

**What the dandiset holds.** The behaviour of 26 rats: 16,115 sessions in 11 protocols, each with a
trials table (context, outcome, generative evidence strengths, and the time of every pulse) and,
in some TaskSwitch6 sessions, optogenetic epochs. The electrophysiology (3,495 units) is not in it.
Files are streamed from the archive: each file is read whole into memory and opened there.

**Outside inputs.** The paper's data repository
[Brody-Lab/flexible_decision_making_rats](https://github.com/Brody-Lab/flexible_decision_making_rats)
at commit `0ea8fdb7c56d755d99155171cc314dbbc24393ac`: its per-session behavioural tables (to test
the link to the dandiset), its trial counts and saved regression weights (as reference values),
and its neural data and fitted per-neuron pulse kernels (for the neural claims, which the dandiset
cannot test). Everything is fetched from GitHub at that commit, into memory.

| Claim | Data | Tested |
|---|---|---|
| session-link | with inputs | yes |
| subjects | dandiset | yes |
| trials-per-session | dandiset | yes |
| blocks-per-session | dandiset | yes |
| session-duration | dandiset | yes |
| block-min-30 | dandiset | yes |
| fig1c-20-rats | dandiset | yes |
| fig1c-context-selection | dandiset | yes |
| ed1e-switch-four-trials | dandiset | yes |
| ed2g-learning-2-5-months | dandiset | yes |
| fig4-converging | dandiset | yes |
| fig4-loc-frq-uncorrelated | dandiset | yes |
| ed3g-halfsplit | dandiset | yes |
| ed4-bilateral-fof-impairs | dandiset | yes |
| neural-link | with inputs | yes |
| units-per-rat | elsewhere (GitHub) | yes |
| fig3h-neural-uncorrelated | elsewhere (GitHub) | yes |
| choice-axes-parallel | elsewhere (GitHub) | yes |
| fig5d-behaviour-neural | with inputs | yes |
| fig1e-choice-axis | elsewhere (GitHub) | blocked: methods underspecified |
| fig1f-irrelevant-encoded | elsewhere (GitHub) | blocked: not runnable |
| fig3a-trained-rnn-svm | elsewhere (GitHub) | blocked: not runnable |
| fig5c-engineered-rnn | elsewhere (GitHub) | blocked: not runnable |

The claims and their criteria are in `claims/001550/10.1038-s41586-024-08433-6.yaml`; the method
choices common to all claims are in its opening comment and in the docstring of `taskswitch.py`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import taskswitch as ts

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = ts.asset_table()
task_assets = assets[assets.protocol.isin(ts.TASK_PROTOCOLS)].reset_index(drop=True)
results = ts.load_many(task_assets.url)
other_assets = assets[assets.protocol.str.startswith("TaskSwitch") & ~assets.protocol.isin(ts.TASK_PROTOCOLS)].reset_index(drop=True)
other_results = ts.load_many(other_assets.url, keep_bins=False)
print(f"dandiset {ts.DANDISET_ID} version {ts.DANDISET_VERSION}: {len(assets)} files; "
      f"read {len(results)} task sessions and {len(other_results)} other TaskSwitch sessions "
      f"in {time.time() - started:.0f} s")


def session_table(table, res):
    s = pd.DataFrame([r["session"] for r in res])
    return pd.concat([table[["subject", "protocol", "session", "date", "size"]].reset_index(drop=True), s], axis=1)


S = session_table(task_assets, results)
S_other = session_table(other_assets, other_results)
S["readable"] = S.missing.fillna("") == ""
S["selected"] = S.readable & (S.valid >= ts.MIN_VALID) & (S.hit >= ts.MIN_HIT) & (S.fsi >= ts.MIN_FSI)

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files per protocol:")
print(assets.protocol.value_counts().to_string())
print("\nSubjects:", assets.subject.nunique(), "| files per subject, min to max:",
      assets.subject.value_counts().min(), "to", assets.subject.value_counts().max())
print("Total size of the task-protocol files:", round(task_assets["size"].sum() / 1e9, 1), "GB")
print("\nTask sessions without one of the required trial columns:", int((~S.readable).sum()))
print(S.loc[~S.readable, ["subject", "session", "missing", "n_trials"]].to_string(index=False))
ok = S[S.readable]
print("\nSessions with a hit_history column:", int(ok.has_hit_history.sum()), "of", len(ok),
      "| sessions where it disagrees with the outcome derived from result_history:",
      int((ok.hit_history_mismatch > 0).sum()))
print("dtype of HistorySection_gammafreq_history:", ok.hist_gfreq_dtype.value_counts().to_dict(),
      "| sessions where it differs from gamma_freq:", int((ok.hist_gfreq_mismatch > 0).sum()))
print("Sex:", ok.sex.value_counts().to_dict(), "| strain:", ok.strain.value_counts().to_dict())

Thirteen task sessions lack the result codes or a pulse column, and are left out of every count
below. Where both exist, `hit_history` agrees with the outcome derived from `result_history` on
every trial, so the outcome is taken from `result_history`, which every other session has. Some
files store `HistorySection_gammafreq_history`, a signed strength, as an unsigned or 16-bit
integer; in every one of them its values equal the float `gamma_freq`, so no sign was lost.

In [ ]:
trials_all = pd.concat([r["trials"].assign(sid=i) for i, r in enumerate(results) if r["trials"] is not None],
                       ignore_index=True)
valid_all = trials_all[trials_all.hit.notna()]
print("Trials:", len(trials_all), "| valid:", len(valid_all),
      "| result codes:", trials_all.result.value_counts().sort_index().to_dict())
print("Context labels:", trials_all.task.value_counts().to_dict(), "| side labels:", trials_all.side.value_counts().to_dict())
print("gamma_dir values (valid trials):", sorted(valid_all.gdir.unique().round(3).tolist()))
print("gamma_freq values (valid trials):", sorted(valid_all.gfreq.unique().round(3).tolist()))
print("duration (s) of the pulse train, valid trials:", valid_all.duration.value_counts().head(5).to_dict())
loose = S[S.readable & S.loose]
print("\nSessions whose pulse times were binned:", len(loose))
print("Pulse times over valid trials (s): min", round(loose.pulse_min.min(), 5), "max", round(loose.pulse_max.max(), 5),
      "| pulses outside 0 to 1.3 s:", int(loose.pulses_outside.sum()))
print("Pulses per valid trial: median", round(loose.pulses_per_trial.median(), 2),
      "range", round(loose.pulses_per_trial.min(), 2), "to", round(loose.pulses_per_trial.max(), 2),
      f"(the Methods' 40 Hz over 1.3 s gives {40 * 1.3:.0f})")

Pulse times lie within 0 to 1.3 s of the start of the pulse train, and there are about 52 per
valid trial, the 40 Hz over 1.3 s of the Methods. The generative strengths take six values,
plus 0 for the training stages that lack one feature. The pulse columns' description says
seconds from the centre poke; the optogenetic epochs below start at `cpoke_start_time` or 0.65 s
after it, which is consistent with the pulse train starting at the centre poke.

In [ ]:
# Do the pulse columns, the correct side and the choice describe the same trials? Contrast the
# true pairing with each trial's pulses paired with the previous trial's side and choice.
rng = np.random.default_rng(1)
rows = []
for i in rng.choice(np.flatnonzero(S.selected), 200, replace=False):
    r = results[i]
    tr = r["trials"][r["trials"].hit.notna()].reset_index(drop=True)
    p = r["pulses"].sum(2).astype(int)                 # right_hi, left_hi, right_lo, left_lo
    loc = (p[:, 0] + p[:, 2]) - (p[:, 1] + p[:, 3])
    frq = (p[:, 0] + p[:, 1]) - (p[:, 2] + p[:, 3])
    rel = np.where(tr.task == "d", loc, frq)
    right = (tr.side == "r").values
    keep = rel != 0
    rows.append(dict(side_true=np.mean((rel[keep] > 0) == right[keep]),
                     side_shift=np.mean((rel[1:][keep[1:]] > 0) == right[:-1][keep[1:]]),
                     choice_true=np.mean((rel[keep] > 0) == (tr.choice.values[keep] == 1)),
                     choice_shift=np.mean((rel[1:][keep[1:]] > 0) == (tr.choice.values[:-1][keep[1:]] == 1)),
                     gdir_sign=np.mean(np.sign(loc[tr.gdir != 0]) == np.sign(tr.gdir[tr.gdir != 0]))))
chk = pd.DataFrame(rows)
print("200 selected sessions; fraction of valid trials (relevant pulse difference not 0) where:")
print(chk.describe().loc[["mean", "min", "max"]].round(3).to_string())

# Optogenetic epochs against the trial clock.
eps = []
for i, r in enumerate(results):
    if r["opto"] is not None and r["trials"] is not None:
        eps.append(ts.opto_trials(r["trials"], r["opto"]).assign(subject=S.subject[i], sid=i))
eps = pd.concat(eps, ignore_index=True)
print("\nOptogenetic epochs:", len(eps), "in", eps.sid.nunique(), "sessions of", sorted(eps.subject.unique()))
print("Epochs whose start falls inside a trial:", int(eps.in_trial.sum()))
print("Start minus cpoke_start_time (s), rounded:", eps.offset.round(3).value_counts().head(6).to_dict())
print("Epoch length (s):", eps.duration.value_counts().to_dict())
print("Sites:", eps.sites.value_counts().to_dict(), "| power (mW):", eps.power.value_counts().to_dict(),
      "| wavelength (nm):", eps.wavelength.value_counts().to_dict())

In selected sessions, the side with more relevant pulses is the stored correct side on every
trial where the counts differ, and the choice follows the relevant pulses on most trials; paired
with the previous trial instead, both fall to about one half. So the pulse columns, the correct
side and the derived choice refer to the same trials, and the sign of the location pulse
difference agrees with `gamma_dir`. Every optogenetic epoch starts inside a trial, at the centre
poke or 0.65 s after it, and lasts 1.3 or 0.65 s, as the column description of `opto_type` says
(full trial, first or second half). The files give the wavelength as 473 nm; Extended Data Figure
4 says 450 nm. Epochs are in four rats, where the paper reports three.

In [ ]:
sel = S[S.selected]
print("Sessions that pass the Methods selection (>= 100 valid trials, >= 0.7 correct, feature selection index >= 0.7):",
      len(sel), "of", int(S.readable.sum()))
print(sel.protocol.value_counts().to_dict())
per_rat = S[S.readable].groupby("subject").agg(sessions=("session", "size"), selected=("selected", "sum"),
                                              valid=("valid", "sum"))
per_rat["valid_selected"] = sel.groupby("subject").valid.sum()
print(per_rat.astype(int).to_string())

## session-link

> The dandiset's sessions can be linked to the paper's published per-session behavioural tables,
> and the linked records agree on trial counts, outcomes, evidence and pulse timing.

In [ ]:
rng = np.random.default_rng(101)
rats = sorted(assets.subject.unique())
blobs = {}
with ThreadPoolExecutor(8) as ex:
    for rat, b in zip(rats, ex.map(lambda r: ts.paper_file(f"data/Sessions_trials/{r}_sessions.mat"), rats)):
        blobs[rat] = b
from scipy.io import loadmat
import io as _io
paper_rows = []
for rat in rats:
    m = loadmat(_io.BytesIO(blobs[rat]), squeeze_me=True, struct_as_record=False, variable_names=["sessions"])
    for k, s in enumerate(np.atleast_1d(m["sessions"])):
        paper_rows.append(dict(subject=rat, k=k, file=s.file, session=ts.paper_session_name(s.file),
                               valid_p=s.valid, hit_p=s.hit, good=s.good, bdir=s.bdir, bfreq=s.bfreq))
Pp = pd.DataFrame(paper_rows)
Pp["has_valid"] = Pp.valid_p.notna() & (Pp.valid_p > 0)
Pp["selected_p"] = (Pp.hit_p >= 0.7) & ((Pp.bdir + Pp.bfreq) / 2 >= 0.7) & (Pp.valid_p >= 100) & (Pp.good == 1)
M = Pp.merge(S[S.readable], on=["subject", "session"], how="outer", indicator=True)
print("Paper sessions:", len(Pp), "| with valid trials:", int(Pp.has_valid.sum()),
      "| found in the dandiset (readable):", int((M._merge == "both").sum()))
found = M[(M._merge != "right_only") & M.has_valid.fillna(False)]
frac_found = (found._merge == "both").mean()
print(f"Paper sessions with valid trials found in the dandiset: {frac_found:.4f}")
print("Paper sessions not found:", found.loc[found._merge == "left_only", ["subject", "session", "valid_p"]].to_string(index=False))
both = M[(M._merge == "both") & M.has_valid.fillna(False)]
eq_valid = (both.valid_p == both.valid).mean()
eq_hit = np.isclose(both.hit_p, both.hit).mean()
rel_ok = np.isclose(both.bdir, both.rel_loc, atol=1e-6) & np.isclose(both.bfreq, both.rel_frq, atol=1e-6)
print(f"Matched sessions: {len(both)}; valid trials equal in {eq_valid:.4f}, fraction correct equal in {eq_hit:.4f}, "
      f"both relative weights within 1e-6 in {rel_ok.mean():.4f}")
only_d = M[M._merge == "right_only"]
print("\nTask sessions in the dandiset and not in the paper's tables:", len(only_d), "| of them pass the selection:",
      int(only_d.selected.sum()))
print(only_d[["subject", "session", "valid", "hit", "fsi", "selected"]].round(3).to_string(index=False))
extra = M[(M._merge == "both") & M.selected.astype(bool) & ~M.selected_p.fillna(False).astype(bool)]
print("\nSelected here, in the paper's tables, not selected there:")
print(extra[["subject", "session", "valid", "hit", "fsi", "valid_p", "hit_p", "good", "bdir", "bfreq", "rel_loc", "rel_frq"]].round(4).to_string(index=False))
xs = pd.crosstab(M.selected.fillna(False).rename("dandiset selection"), M.selected_p.fillna(False).rename("paper selection with good flag"))
print("\n", xs.to_string())

In [ ]:
# Per-trial comparison on a seeded sample of 20 selected sessions found in both.
cand = M[(M._merge == "both") & M.selected & M.selected_p.fillna(False)]
sample = cand.iloc[rng.choice(len(cand), 20, replace=False)]
url_of = dict(zip(task_assets.subject + "/" + task_assets.session, task_assets.url))
rows = []
for _, row in sample.iterrows():
    tr_p = loadmat(_io.BytesIO(blobs[row.subject]), squeeze_me=True, struct_as_record=False,
                   variable_names=["trials"])["trials"][int(row.k)]
    d = ts.load_session(url_of[f"{row.subject}/{row.session}"], bin_s=0.01, n_bins=130, all_bins=True)
    tr = d["trials"][d["trials"].hit.notna()].reset_index(drop=True)
    p = d["pulses"].astype(int)
    mine = dict(vecr=p[:, 0] + p[:, 2], vecl=p[:, 1] + p[:, 3], vechi=p[:, 0] + p[:, 1], veclo=p[:, 2] + p[:, 3])
    n = len(tr)
    same_n = n == len(np.atleast_1d(tr_p.choice_all))
    agree = (same_n
             and np.array_equal(list(str(tr_p.task_all)), tr.task.tolist())
             and np.array_equal(list(str(tr_p.side_all)), tr.side.tolist()))
    per_trial = np.ones(n, bool) if same_n else np.zeros(n, bool)
    if same_n:
        per_trial &= np.array(list(str(tr_p.task_all))) == tr.task.values
        per_trial &= np.array(list(str(tr_p.side_all))) == tr.side.values
        per_trial &= np.isclose(tr_p.gdir_all, tr.gdir) & np.isclose(tr_p.gfreq_all, tr.gfreq)
        per_trial &= np.atleast_1d(tr_p.choice_all) == tr.choice.values
        per_trial &= np.atleast_1d(tr_p.hits_all) == tr.hit.values
        nta_eq = np.mean(np.atleast_1d(tr_p.nta_all) == tr.nta.values)
        bins_eq = np.mean([np.mean(getattr(tr_p, f"{k}_all") == mine[k]) for k in mine])
    else:
        nta_eq = bins_eq = np.nan
    rows.append(dict(subject=row.subject, session=row.session, trials=n, same_count=same_n,
                     trials_agree=per_trial.mean(), nta_agree=nta_eq, bins_10ms_agree=bins_eq))
sample_cmp = pd.DataFrame(rows)
print(sample_cmp.round(5).to_string(index=False))
print("\nOver the sample: trials agreeing on context, side, strengths, choice and outcome:",
      round(np.average(sample_cmp.trials_agree, weights=sample_cmp.trials), 5),
      "| position in block:", round(np.average(sample_cmp.nta_agree, weights=sample_cmp.trials), 5),
      "| 10 ms pulse bins:", round(np.average(sample_cmp.bins_10ms_agree, weights=sample_cmp.trials), 5))

num = ts.paper_mat("data/numsessions_all_rats.mat")
numhits = pd.Series(num["numhits"], index=num["rats"])
cmp_hits = pd.DataFrame(dict(paper_numhits=numhits, dandiset=per_rat.valid_selected)).fillna(0).astype(int)
cmp_hits["difference"] = cmp_hits.dandiset - cmp_hits.paper_numhits
print("\nValid trials in selected sessions, per rat, against the paper's numsessions_all_rats.mat:")
print(cmp_hits.to_string())
print("Rats with equal counts:", int((cmp_hits.difference == 0).sum()), "of", len(cmp_hits))
del blobs

**Result: reproduced.** Of the 15,278 paper sessions with valid trials, 15,277 (0.9999) are in
the dandiset; the one missing (P189 TaskSwitch6-210516a) has 27 valid trials. In all 15,277
matched sessions the number of valid trials and the fraction correct are equal. In the 20 sampled
sessions every valid trial agrees on context, correct side, strengths, choice and outcome, the
position in block agrees on every trial, and 0.99996 of the 10 ms pulse bins agree. The Methods
selection made from the dandiset alone keeps all 8,843 sessions the paper's code keeps, with its
`good` flag, plus 6 more: 5 P187 and P189 sessions absent from the paper's tables, and P059
TaskSwitch4-170331a, where the location weight in the location context is 1.112 here and 1.0 in
the paper's table, which caps it, so its index falls to 0.697 there. The per-rat trial counts
equal the paper's for 23 of 26 rats and exceed them by 213, 1,442 and 609 trials for P059, P187
and P189.

## subjects

> The study used 26 male Long-Evans rats, aged between 6 and 24 months.

In [ ]:
dob = S[S.readable].groupby("subject").date_of_birth.agg(lambda x: sorted(set(x)))
print("Subjects:", assets.subject.nunique(), "| subjects with more than one date of birth:", int((dob.str.len() > 1).sum()))
age = (pd.to_datetime(sel.session_start.str[:10]) - pd.to_datetime(sel.date_of_birth.str[:10])).dt.days / 30.44
print(f"Age at the {len(sel)} selected sessions (months): min {age.min():.1f}, median {age.median():.1f}, max {age.max():.1f}")
inside = ((age >= 5.4) & (age <= 26.4)).mean()
print(f"Fraction within 5.4 to 26.4 months: {inside:.4f}; within 6 to 24: {((age >= 6) & (age <= 24)).mean():.4f}")
by_rat = pd.DataFrame(dict(min=age.groupby(sel.subject).min(), max=age.groupby(sel.subject).max())).round(1)
print(by_rat.T.to_string())
print("Rats with a selected session below 5.4 months:", int((by_rat["min"] < 5.4).sum()), "| above 26.4 months:", int((by_rat["max"] > 26.4).sum()))

**Result: partially reproduced.** The dandiset holds 26 subjects, all male Long Evans. Ages at
the 8,849 selected sessions run from 3.0 to 39.1 months (median 15.4); 0.913 of sessions lie
within the widened 5.4 to 26.4 months and 0.844 within 6 to 24. Eighteen rats have selected
sessions before 5.4 months (the youngest at 3.0) and eight after 26.4 (P189 up to 39.1).

## trials-per-session, blocks-per-session, session-duration

> Rats performed on average 542 trials per session. On average, rats switched across 14.6 context
> blocks per session. Behavioural sessions lasted 2 to 4 hours.

In [ ]:
per_session = trials_all.groupby("sid").agg(trials=("hit", "size"), valid=("hit", "count"),
                                             blocks=("block", "max"), first=("start", "min"), last=("stop", "max"))
per_session["blocks"] += 1
per_session["switches"] = per_session.blocks - 1
per_session["hours"] = (per_session["last"] - per_session["first"]) / 3600
per_session["selected"] = S.selected.reindex(per_session.index).values
groups = {"selected sessions": per_session[per_session.selected], "all task sessions": per_session}
rows = []
for name, g in groups.items():
    rows.append(dict(sessions=name, n=len(g), mean_trials=g.trials.mean(), mean_valid=g.valid.mean(),
                     mean_blocks=g.blocks.mean(), mean_switches=g.switches.mean(),
                     median_hours=g.hours.median(), frac_2_to_4h=((g.hours >= 2) & (g.hours <= 4)).mean(),
                     frac_1p8_to_4p4h=((g.hours >= 1.8) & (g.hours <= 4.4)).mean(),
                     hours_5_95=f"{g.hours.quantile(.05):.2f} to {g.hours.quantile(.95):.2f}"))
summary_sessions = pd.DataFrame(rows).set_index("sessions")
print(summary_sessions.round(3).T.to_string())
print("\nPaper: 542 trials (tolerance 487.8 to 596.2); 14.6 blocks (13.14 to 16.06); 2 to 4 h (median within 1.8 to 4.4 h)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
g = per_session[per_session.selected]
ax[0].hist(g.trials, 60); ax[0].set_xlabel("trials per selected session")
ax[1].hist(g.blocks, np.arange(0, 60)); ax[1].set_xlabel("context blocks per selected session")
ax[2].hist(g.hours.clip(0, 8), 60); ax[2].set_xlabel("session duration (h, clipped at 8)")
plt.tight_layout(); plt.show()
print("Ranges: trials", g.trials.min(), "to", g.trials.max(), "| blocks", g.blocks.min(), "to", g.blocks.max(),
      "| hours", round(g.hours.min(), 2), "to", round(g.hours.max(), 2), "; sessions above 8 h:", int((g.hours > 8).sum()))

**Result: trials-per-session reproduced.** Valid trials per session average 551.8 over the
selected sessions (reading b) and 494.2 over all task sessions (reading d), both within 487.8 to
596.2; all trials average 817.9 and 749.7 (readings a and c), outside.

**Result: blocks-per-session reproduced.** The number of switches averages 15.98 over the
selected sessions and the number of blocks 13.90 over all task sessions, both within 13.14 to
16.06; blocks over selected sessions (16.98) and switches over all sessions (12.90) are outside.

**Result: session-duration reproduced.** The median duration is 2.50 h over the selected sessions
and 2.57 h over all task sessions, within 1.8 to 4.4 h. Only 0.559 and 0.556 of sessions last 2 to
4 h, and 5% to 95% of selected sessions span 1.26 to 4.41 h.

## block-min-30

> Block switches occurred after a minimum of 30 trials per block, and when a local estimate of
> performance reached 80% correct.

In [ ]:
rows = []
for i in np.flatnonzero(S.selected):
    tr = results[i]["trials"]
    last = tr.block.max()
    for b, g in tr[tr.block < last].groupby("block"):
        v = g.hit.dropna()
        nxt = tr[tr.block == b + 1]
        rows.append(dict(sid=i, all=len(g), valid=len(v), perf_last30=v.tail(30).mean()))
blocks = pd.DataFrame(rows)
print("Non-final blocks in selected sessions:", len(blocks))
print(f"Blocks with >= 30 trials (all trials): {(blocks['all'] >= 30).mean():.4f}; (valid trials): {(blocks.valid >= 30).mean():.4f}")
print("Block length, all trials: percentiles 1, 5, 50:", blocks["all"].quantile([.01, .05, .5]).tolist(),
      "| valid trials:", blocks.valid.quantile([.01, .05, .5]).tolist())
print("Fraction correct over the last 30 valid trials before a switch: median", round(blocks.perf_last30.median(), 3),
      "| fraction at or above 0.8:", round((blocks.perf_last30 >= 0.8).mean(), 3))

**Result: reproduced.** Counted over all trials, every one of the 141,359 non-final blocks in the
selected sessions has at least 30 trials (fraction 1.0000; 1st percentile 30, median 44);
counted over valid trials, 0.817 do. Over the last 30 valid trials before a switch the median
fraction correct is 0.828, and 0.601 of blocks are at or above 0.8, so the 80% rule is not
reached on that window in 40% of blocks; the task's own estimate is not described.

## fig1c-20-rats and fig1c-context-selection

> Twenty rats each contributed more than 120,000 trials after training. After training, choices
> are mostly driven by location in the location context and by frequency in the frequency context.

In [ ]:
counts = per_rat.valid_selected.sort_values(ascending=False)
rats120 = counts[counts > 120000].index.tolist()
print("Rats with more than 120,000 valid trials in selected sessions:", len(rats120),
      "| with at least 120,000 (Figure1c.m uses >=):", int((counts >= 120000).sum()))
print(counts.to_string())

**Result: reproduced.** Twenty rats have more than 120,000 valid trials in the selected sessions
(P189 is the twentieth, with 128,309; P124 is next, with 115,123).

In [ ]:
rows = []
for rat in rats120:
    v = pd.concat([results[i]["trials"] for i in np.flatnonzero(S.selected & (S.subject == rat))])
    v = v[v.hit.notna()]
    rl, rf, fsi = ts.relative_weights(v.choice.values.astype(int), v.task.values, v.gdir.values, v.gfreq.values)
    rows.append(dict(rat=rat, trials=len(v), rel_location_in_LOC=rl, rel_frequency_in_FRQ=rf, index=fsi))
ctx_sel = pd.DataFrame(rows).set_index("rat")
print(ctx_sel.round(3).to_string())
both_ok = (ctx_sel.rel_location_in_LOC > 0.5) & (ctx_sel.rel_frequency_in_FRQ > 0.5)
print("\nRats with both relative weights above 0.5:", int(both_ok.sum()), "of", len(ctx_sel),
      "| lowest:", round(ctx_sel[["rel_location_in_LOC", "rel_frequency_in_FRQ"]].min().min(), 3))

**Result: reproduced.** In all 20 rats the relevant feature carries more than half of the weight
in both contexts: the relative weight of location in the location context is 0.770 to 0.860 and
of frequency in the frequency context 0.724 to 0.870.

## ed1e-switch-four-trials

> Rats switch the feature they attend to within about four trials of a new context block.

In [ ]:
def switch_curves(mask):
    rows = []
    for rat in sorted(S.subject.unique()):
        ids = np.flatnonzero(mask & (S.subject == rat))
        if not len(ids):
            continue
        v = pd.concat([results[i]["trials"] for i in ids])
        v = v[v.hit.notna() & (v.block > 0)]
        for new in ("f", "d"):
            for k in range(1, 31):
                g = v[(v.task == new) & (v.nta == k)]
                if len(g) < 30:
                    continue
                X = np.column_stack([np.ones(len(g)), g.gdir, g.gfreq])
                w = ts._irls(X, g.choice.values.astype(float))
                rel = w[2] / (w[1] + w[2]) if new == "f" else w[1] / (w[1] + w[2])
                rows.append(dict(rat=rat, new=new, k=k, rel=rel, n=len(g)))
    return pd.DataFrame(rows)


def switch_summary(c):
    m = c.groupby(["new", "k"]).rel.mean().unstack(0)
    plateau = m.loc[10:30].mean()
    return m, plateau


main_mask = S.selected
code_mask = S.readable & (S.valid >= 100) & (S.hit >= 0.65) & (S.fsi >= 0.65)
for name, mask in (("Methods selection (main)", main_mask), ("code's 0.65 selection (variant)", code_mask)):
    c = switch_curves(mask)
    m, plateau = switch_summary(c)
    print(f"{name}: rats {c.rat.nunique()}; relevant relative weight by position in the new block "
          f"(new block f = switch from location to frequency):")
    print(m.loc[[1, 2, 3, 4, 5, 6, 8, 10, 20, 30]].round(3).T.to_string())
    print("plateau (positions 10 to 30):", plateau.round(3).to_dict())
    for k in (4, 5):
        print(f"  position {k}: fraction of plateau", (m.loc[k] / plateau).round(3).to_dict())
    if name.startswith("Methods"):
        switch_main = (c, m, plateau)
c, m, plateau = switch_main
fig, ax = plt.subplots(figsize=(5, 3))
for new, lab in (("f", "location to frequency"), ("d", "frequency to location")):
    ax.plot(m.index, m[new], label=lab)
ax.set_xlabel("position in new block (all trials)"); ax.set_ylabel("relevant relative weight"); ax.legend(); plt.show()
print("Plotted range:", round(m.min().min(), 3), "to", round(m.max().max(), 3))

**Result: reproduced.** At position 4 of a new block the across-rat relative weight of the newly
relevant feature is 0.91 (frequency to location) and 0.963 (location to frequency) of its plateau
under the Methods selection, and 0.914 and 0.961 under the code's looser selection. At position 1
it is 0.502 and 0.560, against plateaus of 0.854 and 0.789.

## ed2g-learning-2-5-months

> Most rats learned the task in two to five months.

In [ ]:
all_sessions = pd.concat([S[S.readable], S_other[S_other.missing.fillna("") == ""]], ignore_index=True)
all_sessions["day"] = pd.to_datetime(all_sessions.date, format="%y%m%d")
assets["day"] = pd.to_datetime(assets.date, format="%y%m%d")
start_any = assets.groupby("subject").day.min()
start_ts = assets[assets.protocol.str.startswith("TaskSwitch")].groupby("subject").day.min()
crit_a = all_sessions[(all_sessions.valid >= 100) & (all_sessions.fsi >= 0.7)].groupby("subject").day.min()
crit_b = all_sessions[(all_sessions.valid >= 100) & (all_sessions.fsi >= 0.7) & (all_sessions.hit >= 0.7)].groupby("subject").day.min()
learn = pd.DataFrame({f"{s}_{c}": (cv - sv).dt.days / 30.44
                      for s, sv in (("i_first_session", start_any), ("ii_first_taskswitch", start_ts))
                      for c, cv in (("a_index", crit_a), ("b_selection", crit_b))})
print(learn.round(2).to_string())
print("\nFraction of rats within 1.8 to 5.5 months, per reading:")
print(((learn >= 1.8) & (learn <= 5.5)).sum().astype(str) + " of " + learn.notna().sum().astype(str))
print("Within 2 to 5 months:", (((learn >= 2) & (learn <= 5)).sum()).to_dict())
print("Protocols of each rat's first session:", assets.sort_values("day").groupby("subject").protocol.first().value_counts().to_dict())

**Result: reproduced.** Counting from each rat's first session and taking the first session with
an index of at least 0.7 (readings i and a), 17 of 26 rats took 1.8 to 5.5 months; with the full
selection (i, b) 19 of 26, and from the first TaskSwitch session 13 (ii, a, exactly half) and 17
(ii, b). Within the unwidened 2 to 5 months the counts are 12, 16, 10 and 13. Times under reading
(i, a) run from 0.85 to 8.18 months.

## Behavioural kernels

The kernels of fig4-converging, fig4-loc-frq-uncorrelated, ed3g-halfsplit and
fig5d-behaviour-neural are fitted here. The main fit uses the code's smoothness penalty with its
weight chosen per rat and context by 3-fold cross-validation over 20, 200 and 2000; variants use
the code's fixed 200, and for the recorded rats the code's looser selection (0.65 correct, 0.6
index).

In [ ]:
def rat_data(rat, mask):
    X, y, task = [], [], []
    for i in np.flatnonzero(mask & (S.subject == rat)):
        r = results[i]
        if r["pulses"] is None:
            raise RuntimeError(f"pulses not kept for {S.session[i]}")
        tr = r["trials"][r["trials"].hit.notna()]
        X.append(ts.evidence_features(r["pulses"]))
        y.append(tr.choice.values)
        task.append(tr.task.values)
    return np.vstack(X), np.concatenate(y), np.concatenate(task)


rats100 = counts[counts >= 100000].index.tolist()
rats_fit = sorted(set(rats100) | set(ts.PAPER_EPHYS_RATS))
code_ephys_mask = S.readable & (S.valid >= 100) & (S.hit >= 0.65) & (S.fsi >= 0.6)
t0 = time.time()


def fit_rat(rat):
    seed = int(rat[1:]) + 1000                 # each rat's folds have their own seed
    X, y, task = rat_data(rat, S.selected)
    out = dict(rat=rat, n=len(y), main=ts.kernels(X, y, task, seed=seed))
    out["fixed200"] = ts.kernels(X, y, task, lam=200.0)
    if rat in rats100:
        halves = []
        for h in (0, 1):
            Xh, yh, th = [], [], []
            for ctx in ("d", "f"):
                idx = np.flatnonzero(task == ctx)
                cut = int(round(len(idx) / 2))
                part = idx[:cut] if h == 0 else idx[cut:]
                Xh.append(X[part]); yh.append(y[part]); th.append(task[part])
            Xh, yh, th = np.vstack(Xh), np.concatenate(yh), np.concatenate(th)
            halves.append((ts.kernels(Xh, yh, th, lam={c: out["main"][f"lam_{c}"] for c in "df"}),
                           ts.kernels(Xh, yh, th, lam=200.0)))
        out["halves"] = halves
    if rat in ts.PAPER_EPHYS_RATS:
        Xc, yc, tc = rat_data(rat, code_ephys_mask)
        out["code_sel_n"] = len(yc)
        out["code_sel_200"] = ts.kernels(Xc, yc, tc, lam=200.0)
    return out


with ThreadPoolExecutor(4) as ex:
    fits = {f["rat"]: f for f in ex.map(fit_rat, rats_fit)}
print(f"fitted {len(fits)} rats in {time.time() - t0:.0f} s; every fit converged (a failed fit raises)")
print(pd.DataFrame({r: dict(trials=f["n"], lam_LOC=f["main"]["lam_d"], lam_FRQ=f["main"]["lam_f"])
                    for r, f in fits.items()}).T.to_string())

## fig4-converging

> Differential behavioural kernels predominantly converge towards the end of the pulse train, that
> is the difference between contexts in the weight of a feature shrinks for later pulses.

In [ ]:
rats18 = [r for r in rats120 if int(r[1:]) >= 15]
print("Figure 4 rats (> 120,000 valid trials, not numbered below 15):", len(rats18), rats18)
saved = {}
for rat in rats18:
    p = ts.paper_mat(f"data/logistic_regression_results/logreg_best_{rat}_5.mat")
    saved[rat] = ts.slopes(dict(dd=p["thetad"][1:27], fd=p["thetad"][27:], df=p["thetaf"][1:27], ff=p["thetaf"][27:]))
sl = pd.DataFrame({r: dict(loc=ts.slopes(fits[r]["main"])[0], frq=ts.slopes(fits[r]["main"])[1],
                           loc_200=ts.slopes(fits[r]["fixed200"])[0], frq_200=ts.slopes(fits[r]["fixed200"])[1],
                           loc_paper_weights=saved[r][0], frq_paper_weights=saved[r][1]) for r in rats18}).T
print(sl.round(3).to_string())
for a, b in (("loc", "frq"), ("loc_200", "frq_200")):
    v = np.r_[sl[a], sl[b]]
    print(f"{a[:3]}+{b[:3]} ({'main' if a == 'loc' else 'fixed 200'}): negative {int((v < 0).sum())} of {len(v)}, "
          f"median {np.median(v):.3f}, range {v.min():.3f} to {v.max():.3f}, within -0.8 to -0.15: {int(((v >= -0.8) & (v <= -0.15)).sum())}")
mine = np.r_[sl.loc_200, sl.frq_200]; theirs = np.r_[sl.loc_paper_weights, sl.frq_paper_weights]
print(f"Fixed-200 slope indices against those from the paper's saved weights: largest absolute difference "
      f"{np.max(np.abs(mine - theirs)):.4f}, r = {np.corrcoef(mine, theirs)[0, 1]:.5f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
lo, hi = np.inf, -np.inf
for rat in rats18:
    k = fits[rat]["main"]
    for a, (rel, irr) in zip(ax, (("dd", "df"), ("ff", "fd"))):
        d = (k[rel] - k[irr]) / np.max(np.abs(k[rel] - k[irr]))
        a.plot(ts.BIN_CENTRES, d, lw=0.8)
        lo, hi = min(lo, d.min()), max(hi, d.max())
for a, t in zip(ax, ("location", "frequency")):
    a.set_title(f"differential kernel, {t} (normalised)"); a.set_xlabel("time in pulse train (s)")
plt.tight_layout(); plt.show()
print("Plotted range:", round(lo, 3), "to", round(hi, 3))

**Result: reproduced.** All 36 slope indices of the 18 rats are negative (median -0.432, range
-0.774 to -0.097; 35 within the code's axis range of -0.8 to -0.15). With the code's fixed penalty
all 36 are negative as well (median -0.491), and these agree with the indices from the paper's
saved weights to at most 0.0177 (r = 0.99973).

## fig4-loc-frq-uncorrelated

> Across rats, the shape of the differential behavioural kernel for one feature does not predict the
> shape for the other feature.

In [ ]:
def pearson_ci(a, b):
    r, p = stats.pearsonr(a, b)
    z, se = np.arctanh(r), 1 / np.sqrt(len(a) - 3)
    return r, p, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)


for a, b, name in (("loc", "frq", "main"), ("loc_200", "frq_200", "fixed 200"), ("loc_paper_weights", "frq_paper_weights", "paper's saved weights")):
    r, p, lo, hi = pearson_ci(sl[a], sl[b])
    print(f"{name}: r = {r:.3f} (95% CI {lo:.3f} to {hi:.3f}), p = {p:.3g}, n = {len(sl)}")

**Result: reproduced.** Across the 18 rats the location and frequency slope indices are not
significantly correlated: r = 0.180 (95% CI -0.313 to 0.597), p = 0.475; with the fixed penalty
r = 0.092, p = 0.717.

## ed3g-halfsplit

> The behavioural slope index computed from the first half of a rat's trials is correlated with the
> index from the second half (r = 0.58, p = 0.000013).

In [ ]:
rows = []
for rat in rats100:
    (m1, f1), (m2, f2) = fits[rat]["halves"]
    rows.append(dict(rat=rat, loc1=ts.slopes(m1)[0], frq1=ts.slopes(m1)[1], loc2=ts.slopes(m2)[0], frq2=ts.slopes(m2)[1],
                     loc1_200=ts.slopes(f1)[0], frq1_200=ts.slopes(f1)[1], loc2_200=ts.slopes(f2)[0], frq2_200=ts.slopes(f2)[1]))
hs = pd.DataFrame(rows).set_index("rat")
print(hs.round(3).to_string())
for suf, name in (("", "main"), ("_200", "fixed 200")):
    a = np.r_[hs[f"loc1{suf}"], hs[f"frq1{suf}"]]; b = np.r_[hs[f"loc2{suf}"], hs[f"frq2{suf}"]]
    r, p, lo, hi = pearson_ci(a, b)
    print(f"{name}: {len(rats100)} rats, {len(a)} points: r = {r:.3f} (95% CI {lo:.3f} to {hi:.3f}), p = {p:.3g}")
    for f in ("loc", "frq"):
        r1, p1 = stats.pearsonr(hs[f"{f}1{suf}"], hs[f"{f}2{suf}"])
        print(f"   {f} alone: r = {r1:.3f}, p = {p1:.3g}")
print("Paper: r = 0.58 (tolerance 0.522 to 0.638), p = 0.000013")

**Result: reproduced.** Over 24 rats and 48 values the first-half and second-half slope indices
correlate with r = 0.552 (95% CI 0.318 to 0.723), p = 4.7e-05, within 0.522 to 0.638; with the
code's fixed penalty r = 0.577, p = 1.8e-05.

## ed4-bilateral-fof-impairs

> Bilateral optogenetic silencing of the frontal orienting fields during the stimulus impairs
> performance of the task (three rats).

In [ ]:
rows = []
opto_sessions = S[S.readable & (S.valid >= 100)].index.intersection(eps.sid.unique())
for i in opto_sessions:
    tr = results[i]["trials"].reset_index(drop=True)
    e = eps[(eps.sid == i) & eps.in_trial]
    lab = pd.Series("control", index=tr.index, dtype=object)
    for _, ep in e.iterrows():
        lab[ep.trial] = f"{ep.sites}_{ep.window}"
    tr = tr.assign(cond=lab.values)
    tr = tr[tr.hit.notna()]
    rows.append(tr.assign(subject=S.subject[i]))
ot = pd.concat(rows, ignore_index=True)
n_sessions = S.loc[opto_sessions].groupby("subject").size()
print("Opto sessions with >= 100 valid trials per rat:", n_sessions.to_dict())
tab = ot.groupby(["subject", "cond"]).hit.agg(["size", "mean"]).unstack(1)
print(tab["mean"].round(3).to_string()); print(tab["size"].fillna(0).astype(int).to_string())
top3 = n_sessions.sort_values(ascending=False).index[:3].tolist()
res_rows = []
for rat in n_sessions.index:
    g = ot[ot.subject == rat]
    ctl = g[g.cond == "control"].hit
    for name, cond in (("full", g.cond == "bilateral_full"), ("any", g.cond.str.startswith("bilateral"))):
        b = g[cond].hit
        if len(b) == 0:
            res_rows.append(dict(rat=rat, epochs=name, n_bilateral=0)); continue
        table = [[int(b.sum()), int(len(b) - b.sum())], [int(ctl.sum()), int(len(ctl) - ctl.sum())]]
        p = stats.fisher_exact(table)[1]
        res_rows.append(dict(rat=rat, epochs=name, n_bilateral=len(b), acc_bilateral=b.mean(), n_control=len(ctl),
                             acc_control=ctl.mean(), difference=b.mean() - ctl.mean(), p=p))
bil = pd.DataFrame(res_rows)
print("\n", bil.round(4).to_string(index=False))
print("\nThree rats with the most opto sessions:", top3)
for name in ("full", "any"):
    for reading, rr in (("three rats", top3), ("all rats", list(n_sessions.index))):
        g = bil[(bil.epochs == name) & bil.rat.isin(rr)]
        sig = ((g.difference < 0) & (g.p < 0.05)).sum()
        print(f"  {name} bilateral epochs, {reading}: lower with p < 0.05 in {sig} of {len(rr)}")

**Result: partially reproduced.** Four rats have optogenetic sessions, where the paper reports
three. With bilateral full-trial epochs, accuracy is lower than on control trials of the same
sessions with p < 0.05 in 2 of the 3 rats with the most opto sessions: P116 (0.639 against 0.733,
p below 1e-4) and P131 (0.686 against 0.727, p = 0.0001). P124 is not lower (0.712 against 0.687,
p = 0.17), nor is P127 (4 sessions, 0.603 against 0.634, p = 0.55). Any bilateral epoch gives the
same counts.

## neural-link, units-per-rat

> The neurons in the paper's published neural data can be linked to the dandiset's rats and
> sessions; they come from 199 sessions of 7 rats. The recordings yielded 3,495 putative single
> units, 2,047, 832, 258, 94 and 54 from five rats in FOF and 112 and 98 from two rats in mPFC.

In [ ]:
t0 = time.time()
rat_numbers = ts.paper_mat("data/rat_for_each_neuron.mat")["rat_numbers"].astype(int)
n_cells = len(rat_numbers)


def read_cell(n):
    m = ts.paper_mat(f"data/neural_data/cell_{n}.mat", variable_names=["behav", "rat_name", "brain_region"])
    b = m["behav"]
    return dict(cell=n, rat=str(m["rat_name"]), region=str(m["brain_region"]), n_trials=int(b.nTrials),
                sym=ts.trial_symbols(list(str(b.task)), np.atleast_1d(b.gdir), np.atleast_1d(b.gfreq), np.atleast_1d(b.choice)))


with ThreadPoolExecutor(16) as ex:
    cells = pd.DataFrame(list(ex.map(read_cell, range(1, n_cells + 1))))
print(f"read {len(cells)} cell files in {time.time() - t0:.0f} s")
print("Cells per rat and region:")
print(pd.crosstab(cells.rat, cells.region).to_string())
print("Units in total:", len(cells), "| FOF:", int((cells.region == "FOF").sum()), "| mPFC:", int((cells.region == "mPFC").sum()))
print("rat_for_each_neuron agrees with the rat named in each cell file:",
      bool((("P" + pd.Series(rat_numbers).astype(str).str.zfill(3)).values == cells.rat.values).all()))

In [ ]:
# Each rat's valid trials, session by session, as one string; a cell matches where its trials appear.
sym_sessions = {}
for i, r in enumerate(results):
    if r["trials"] is None or S.subject[i] not in ts.PAPER_EPHYS_RATS:
        continue
    v = r["trials"][r["trials"].hit.notna()]
    sym_sessions.setdefault(S.subject[i], []).append((S.session[i], ts.trial_symbols(v.task, v.gdir, v.gfreq, v.choice)))
matches = []
for _, c in cells.iterrows():
    found = [sid for sid, s in sym_sessions.get(c.rat, []) if c.sym and c.sym in s]
    matches.append(found)
cells["n_match"] = [len(f) for f in matches]
cells["session"] = [f[0] if len(f) == 1 else None for f in matches]
print("Rats of the neural data in the dandiset:", sorted(set(cells.rat) & set(assets.subject)))
print("Cells matching exactly one session:", int((cells.n_match == 1).sum()), "of", len(cells),
      f"({(cells.n_match == 1).mean():.4f}); none: {int((cells.n_match == 0).sum())}; more than one: {int((cells.n_match > 1).sum())}")
print("Distinct matched sessions:", cells.session.nunique(), "| per rat:", cells.groupby("rat").session.nunique().to_dict())
print("Cells without a match, per rat:", cells[cells.n_match == 0].groupby("rat").size().to_dict(),
      "| their trial counts:", cells[cells.n_match == 0].n_trials.describe()[["min", "50%", "max"]].to_dict())
print("Matched sessions by protocol:", pd.Series([s.split("-")[0] for s in cells.session.dropna().unique()]).value_counts().to_dict())
print("Paper: 199 sessions (tolerance 179.1 to 218.9)")

# Variant: the cell's trials as an ordered subsequence of one session's valid trials (gaps allowed).
gap = [m if m else ts.gap_match(c.sym, sym_sessions.get(c.rat, [])) for m, (_, c) in zip(matches, cells.iterrows())]
cells["n_gap"] = [len(g) for g in gap]
cells["session_gap"] = [g[0] if len(g) == 1 else None for g in gap]
print("\nVariant, gaps allowed: cells matching exactly one session", int((cells.n_gap == 1).sum()), "of", len(cells),
      f"({(cells.n_gap == 1).mean():.4f}); none: {int((cells.n_gap == 0).sum())}; more than one: {int((cells.n_gap > 1).sum())}")
print("Distinct sessions:", cells.session_gap.nunique(), "| per rat:", cells.groupby("rat").session_gap.nunique().to_dict())
unm = cells[cells.n_gap == 0]
print("Cells still unmatched, per rat:", unm.groupby("rat").size().to_dict())

**Result: neural-link not reproduced.** The seven rats are in the dandiset and every cell names
one of them, but only 2,371 of 3,495 cells (0.678) have trials equal to a contiguous run of one
session's valid trials, from 110 distinct sessions against the paper's 199 (tolerance 179 to
219). With gaps allowed (the variant), 3,269 cells (0.935) match exactly one session and none
matches more than one, so the neural files leave out some of a session's valid trials and the
rule chosen here is stricter than the data need; but these cells still come from only 112
dandiset sessions, and 226 cells, 189 of them P100's, match none.

**Result: units-per-rat reproduced.** The cell files hold 3,495 units, 3,285 in FOF (2,047, 832,
258, 94 and 54 in P100, P059, P055, P095 and P049) and 210 in mPFC (112 in P101 and 98 in P102),
as the Methods' per-rat counts say. The Methods' "3,495 ... recorded from 5 rats in FOF" counts
the mPFC units as well.

## fig3h-neural-uncorrelated, choice-axes-parallel

> Neural slope indices vary across rats and features, with no apparent correlation between the
> location and frequency indices across the seven recorded rats. Choice axes estimated separately
> in the two contexts are essentially parallel (average angle 1.6 degrees).

In [ ]:
ker = ts.paper_mat("data/combined_kernels.mat")["data_all_trials"]
ker_ctx = ts.paper_mat("data/combined_kernels_per_context.mat")
neural = pd.DataFrame({r: dict(zip(("loc", "frq"), ts.neural_slopes(ker, rat_numbers, int(r[1:])))) for r in ts.PAPER_EPHYS_RATS}).T
neural["units"] = [int((rat_numbers == int(r[1:])).sum()) for r in neural.index]
print(neural.round(3).to_string())
r, p, lo, hi = pearson_ci(neural["loc"], neural["frq"])
print(f"\nLOC against FRQ neural slope index across 7 rats: r = {r:.3f} (95% CI {lo:.3f} to {hi:.3f}), p = {p:.3g}")
print("Range: LOC", round(neural["loc"].min(), 3), "to", round(neural["loc"].max(), 3),
      "| FRQ", round(neural["frq"].min(), 3), "to", round(neural["frq"].max(), 3))

**Result: reproduced.** The neural location and frequency slope indices from the paper's fitted
kernels are not significantly correlated across the seven rats (r = -0.177, 95% CI -0.821 to
0.665, p = 0.704), and they vary: location -0.221 to 1.728, frequency 0.465 to 1.500.

In [ ]:
ang = pd.Series({r: ts.choice_axis_angle(ker_ctx["data_all_trials_loc"], ker_ctx["data_all_trials_freq"], rat_numbers, int(r[1:]))
                 for r in ts.PAPER_EPHYS_RATS})
folded = np.minimum(ang, 180 - ang)
good = ~np.isnan(ker_ctx["data_all_trials_loc"].glm_choice.mean(0)) & ~np.isnan(ker_ctx["data_all_trials_freq"].glm_choice.mean(0))
n_used = pd.Series({r: int((good & (rat_numbers == int(r[1:]))).sum()) for r in ts.PAPER_EPHYS_RATS})
print(pd.DataFrame(dict(angle=ang, folded=folded, units_with_both_fits=n_used)).round(2).to_string())
print(f"\nMean angle over 7 rats: {ang.mean():.2f} degrees; folded to at most 90: {folded.mean():.2f}; paper 1.6 (tolerance 1.44 to 1.76)")

**Result: not reproduced.** By the procedure of ExtDataFigure5ag.m on the paper's own fitted
per-context kernels, the angles between the context choice axes are 13.4 to 36.3 degrees in six
rats and 128.2 in P102; the mean is 40.3 degrees (29.4 folded), against the paper's 1.6.

## fig5d-behaviour-neural

> The behavioural slope index and the neural slope index are anti-correlated across rats and
> features (r = -0.73, P < 0.01), and for location (r = -0.71) and frequency (r = -0.71) alone
> (P < 0.1).

In [ ]:
half2 = slice(13, 26)                      # bins 14 to 26, the second half of the stimulus
beh = pd.DataFrame({r: dict(loc=ts.slopes(fits[r]["main"], half2)[0], frq=ts.slopes(fits[r]["main"], half2)[1],
                            loc_code=ts.slopes(fits[r]["code_sel_200"], half2)[0], frq_code=ts.slopes(fits[r]["code_sel_200"], half2)[1],
                            trials=fits[r]["n"], trials_code=fits[r]["code_sel_n"]) for r in ts.PAPER_EPHYS_RATS}).T
saved_e = ts.paper_mat("data/logistic_regression_results_ephys_rats.mat")
beh["loc_paper_saved"], beh["frq_paper_saved"] = saved_e["location"], saved_e["frequency"]
print(pd.concat([beh, neural.add_prefix("neural_")], axis=1).round(3).to_string())


def corr_report(bl, bf, name):
    r, p, lo, hi = pearson_ci(np.r_[bl, bf], np.r_[neural["loc"], neural["frq"]])
    rl, pl = stats.pearsonr(bl, neural["loc"]); rf, pf = stats.pearsonr(bf, neural["frq"])
    print(f"{name}: both r = {r:.3f} (95% CI {lo:.3f} to {hi:.3f}), p = {p:.3g}; location r = {rl:.3f} (p = {pl:.3g}); "
          f"frequency r = {rf:.3f} (p = {pf:.3g})")


corr_report(beh["loc"], beh["frq"], "main (Methods selection, cross-validated penalty)")
corr_report(beh["loc_code"], beh["frq_code"], "variant (code's 0.65/0.6 selection, penalty 200)")
corr_report(beh["loc_paper_saved"], beh["frq_paper_saved"], "reference (paper's saved behavioural indices)")
print("Paper: both -0.73 (tolerance -0.803 to -0.657), P < 0.01; location -0.71, frequency -0.71, P < 0.1")
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.scatter(beh["loc"], neural["loc"], c="k", label="location"); ax.scatter(beh["frq"], neural["frq"], c="b", label="frequency")
ax.set_xlabel("behavioural slope index (dandiset)"); ax.set_ylabel("neural slope index (paper kernels)"); ax.legend(); plt.show()
print("Plotted ranges: behaviour", round(min(beh['loc'].min(), beh['frq'].min()), 3), "to", round(max(beh['loc'].max(), beh['frq'].max()), 3),
      "| neural", round(min(neural['loc'].min(), neural['frq'].min()), 3), "to", round(max(neural['loc'].max(), neural['frq'].max()), 3))

**Result: reproduced.** Between the behavioural slope index from the dandiset (Methods selection,
second half of the stimulus) and the neural slope index from the paper's kernels, r = -0.697 over
14 values (95% CI -0.896 to -0.264), p = 0.0056, within -0.803 to -0.657. For location alone
r = -0.675 (p = 0.096) and for frequency alone r = -0.772 (p = 0.042). With the code's selection
and penalty r = -0.765 (p = 0.0014), and the paper's own saved behavioural indices give -0.768.
P049 and P059 have 23,014 and 25,847 trials under the Methods selection against 135,001 and
162,317 under the code's, and their location indices change sign between the two (0.131 against
-0.432; 1.292 against -0.613).

In [ ]:
print(f"total run time {time.time() - started:.0f} s")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| session-link | with inputs | reproduced |
| subjects | dandiset | partially reproduced |
| trials-per-session | dandiset | reproduced |
| blocks-per-session | dandiset | reproduced |
| session-duration | dandiset | reproduced |
| block-min-30 | dandiset | reproduced |
| fig1c-20-rats | dandiset | reproduced |
| fig1c-context-selection | dandiset | reproduced |
| ed1e-switch-four-trials | dandiset | reproduced |
| ed2g-learning-2-5-months | dandiset | reproduced |
| fig4-converging | dandiset | reproduced |
| fig4-loc-frq-uncorrelated | dandiset | reproduced |
| ed3g-halfsplit | dandiset | reproduced |
| ed4-bilateral-fof-impairs | dandiset | partially reproduced |
| neural-link | with inputs | not reproduced |
| units-per-rat | elsewhere | reproduced |
| fig3h-neural-uncorrelated | elsewhere | reproduced |
| choice-axes-parallel | elsewhere | not reproduced |
| fig5d-behaviour-neural | with inputs | reproduced |
| fig1e-choice-axis | elsewhere | blocked: methods underspecified |
| fig1f-irrelevant-encoded | elsewhere | blocked: not runnable |
| fig3a-trained-rnn-svm | elsewhere | blocked: not runnable |
| fig5c-engineered-rnn | elsewhere | blocked: not runnable |

**How the analysis differs from the paper's.** The session selection is the Methods' (the code's
extra `good` flag is not in the dandiset, and never removes a session that passes the other
criteria). The feature selection index uses the stored generative log rate ratios, which give the
paper's own per-session values. The behavioural kernels use the code's smoothness penalty, read
as the Methods' "ridge regression", with its weight chosen by 3-fold cross-validation over 20,
200 and 2000 (grid and folds chosen here); the code's fixed 200 is shown as a variant and gives
slope indices within 0.018 of those from the paper's saved weights. For the recorded rats the
Methods selection keeps far fewer trials of P049 and P059 than the code's looser one, which is
shown as a variant. Several numbers in the Methods (trials and blocks per session, session
length) do not say which sessions or trials they count; the readings are listed in the claims
file. The neural claims use the paper's fitted per-neuron kernels from its GitHub repository, not
a refit from spikes. The mTDR and RNN analyses were not run.

**What was found about the dandiset.** It holds the behaviour of all 26 rats and links one to one
to the paper's published per-session tables: 15,277 of 15,278 paper sessions with valid trials
are in it, with identical trial counts, outcomes, strengths and pulse timing, and the paper's
trial selection can be rebuilt from it alone. It does not hold the electrophysiology, which is
only in the paper's GitHub repository, as MATLAB files per cell with no session identifier; most
cells can be placed in a dandiset session only by matching their trial sequences. The optogenetic
epochs state 473 nm where the paper says 450 nm, and are present in four rats where the paper
reports three, without saying which three were analysed. Dates of birth give ages of 3 to 39
months at the analysed sessions, where the Methods say 6 to 24. Thirteen task sessions lack the
result codes or a pulse column, and one paper session with 27 valid trials is not in the
dandiset. The cells of the paper's neural files match 112 dandiset sessions, not the 199 the
paper reports, and 226 match none. The choice-axis angles computed from the paper's own kernels by its own code are far
from the 1.6 degrees it reports.